# Lab 52087: Amortized Inference


---

## Task 1 — Amortized GAN inversion and comparison

**1.1 Amortize inversion into an encoder**  
Amortize optimization-based inversion into a network: (i) implement or use **optimization-based inversion** (optimize $z$ so that $G(z)\approx x$). (ii) Train an **encoder** $E$ so that $E(x)$ approximates a good latent (e.g. on pairs $(x,z)$ with $x=G(z)$). At test time, $\hat z = E(x)$ and $\hat x = G(\hat z)$ is the amortized reconstruction.

**1.2 Compare three schemes and report metrics**  
Compare: **(a)** direct optimization-based inversion; **(b)** encoder-only $\hat z = E(x)$; **(c)** encoder + refinement (optimize starting from $E(x)$). Define **your own metrics** (e.g. PSNR, LPIPS) and run on a **small dataset** (e.g. random samples from $G$). Report which method is best and when encoder+refinement approaches direct inversion.

---

## Task 2 (Bonus) — Language-controlled encoding / text-guided generation

Use **text** to control encoding or generation (e.g. text-to-image or text-guided editing). For example: condition the encoder on a text prompt (e.g. CLIP) so that $E(x,\text{prompt})$ yields a latent that follows the prompt; or combine encoder with text-conditioned refinement. Goal: show that text can steer the latent or reconstruction in a meaningful way.



Setup

In [ ]:
# !pip -q install --upgrade huggingface_hub transformers torchvision

import os, sys, time, random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import make_grid

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Load the pretrained StyleGAN generator \(G\)

We use the Hugging Face model `hajar001/stylegan2-ffhq-128`.


In [ ]:
from huggingface_hub import hf_hub_download

model_file = hf_hub_download(
    repo_id="hajar001/stylegan2-ffhq-128",
    filename="style_gan.py"
)
sys.path.insert(0, os.path.dirname(model_file))

from style_gan import StyleGAN

G = StyleGAN.from_pretrained("hajar001/stylegan2-ffhq-128").to(device).eval()

Z_DIM = 512  # latent dimension per model card

def G_generate(z, truncation_psi=0.7):
    x = G.generate(z, truncation_psi=truncation_psi)  # [-1,1]
    x = (x + 1) / 2
    return x.clamp(0, 1)

def show_tensor_images(x, nrow=4, title=None):
    grid = make_grid(x.detach().cpu(), nrow=nrow)
    plt.figure(figsize=(8,8))
    if title: plt.title(title)
    plt.imshow(grid.permute(1,2,0))
    plt.axis("off")
    plt.show()

# sanity check
z = torch.randn(8, Z_DIM, device=device)
x = G_generate(z, truncation_psi=0.7)
show_tensor_images(x, nrow=4, title="Samples from G")


__Task 0.0 — Optimization-based inversion__

Create a synthetic target $x_\text{tgt}=G(z_0)$ and implement per-image inversion: optimize $z$ (e.g. from random init) so that $G(z)\approx x_\text{tgt}$. This is the baseline that you will later amortize into an encoder. (you may use your code from last lab)


In [ ]:
torch.manual_seed(0)
z0 = torch.randn(1, Z_DIM, device=device)
x_tgt = G_generate(z0, truncation_psi=0.7)

show_tensor_images(x_tgt, nrow=1, title="Target image x_tgt = G(z0)")


In [ ]:
# your inversion code here

In [ ]:
plt.figure()
plt.plot(losses_opt)
plt.xlabel("iteration")
plt.ylabel("MSE loss")
plt.title("Optimization-based inversion loss curve")
plt.show()


__Optional: invert a real face image__

You can try optimization-based inversion on a real photo (128×128): set `REAL_IMAGE_PATH` and run the cell. The target may be off the generator manifold, so reconstruction quality can vary.


__Task 1.1 — Encoder $E$ and training__

Define an encoder $E$ that maps image $x$ to latent $\hat z$. Train $E$ on synthetic pairs $(z, x)$ with $x=G(z)$ so that $E(x)\approx z$ (or so that $G(E(x))\approx x$). At test time, $\hat z = E(x)$ is a single forward pass.


In [ ]:
class Encoder(nn.Module):
    # CNN encoder: x (3x128x128) -> z (512)
    def __init__(self, z_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, 4, 1, 0), nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(512, z_dim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.fc(h)

# E = Encoder(Z_DIM).to(device)

# x_tmp = G_generate(torch.randn(2, Z_DIM, device=device))
# print("Shapes:", x_tmp.shape, E(x_tmp).shape)


In [ ]:
# your training code here

__Task 1.2 — Compare three schemes and report metrics__

Implement a comparison of: **(a)** direct optimization-based inversion; **(b)** encoder-only $\hat z = E(x)$; **(c)** encoder + refinement (few optimization steps from $E(x)$).  
Define **your own evaluation metrics** (e.g. PSNR, LPIPS, SSIM, or MSE in pixel/latent space) and evaluate on a **small dataset** (e.g. 16–64 random samples from $G$). Report a short summary or table: which method gives the best reconstruction on your metrics, and how does encoder+refinement compare to direct inversion as you vary the number of refinement steps?


In [ ]:
# metrics: PSNR, LPIPS, SSIM, or MSE in pixel/latent space.

# compare three schemes: 
# 1. direct optimization-based inversion
# 2. encoder-only $\hat z = E(x)$
# 3. encoder + refinement (few optimization steps from $E(x)$) as a warm start

__Task 2 (Bonus) — Language-controlled encoding / text-guided generation__

Use **text** to control the encoding or generation process so that you can achieve a form of **text-to-image** or **text-guided face editing**. For example: condition the encoder on a text prompt (e.g. via CLIP text embeddings) so that $E(x, \text{prompt})$ produces a latent that reconstructs or edits the face according to the prompt; or combine an encoder with a text-conditioned refinement step. You may use the provided CLIP-based setup or consider stronger / different models. The goal is to demonstrate that text can steer the latent or the reconstruction in a meaningful way.


In [ ]:
# Design your own encoder

__Task3 (Not bonus but just try it out a bit, I won't score this one): try to have fun with the Dust3r__
E.g. reconstruct 3D from two images and show the 3D.

In [ ]:
# Enjoy yourself here!